# Flood-It! data export

This notebook gets the data we need to start the adaptive gameplay ML project. Use a BigQuery sandbox project or an existing project where you have permission to run free queries. [Set up the sandbox here](https://docs.cloud.google.com/bigquery/docs/sandbox).

Run the cells in order. Google will ask you to sign in. Use the project ID from your Google Cloud project, not the public dataset's project ID. The notebook estimates the amount of data scanned and asks before running the queries. It will download a ZIP of the CSV results that we can use in the project.


In [ ]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
project_id = input('Enter your Google Cloud project ID: ').strip()
if not project_id:
    raise ValueError('Enter the ID of your own BigQuery project.')
client = bigquery.Client(project=project_id)


In [ ]:
# These are the same queries we keep in the project under sql/.
queries = {'dataset_summary': "-- Get the overall size of the dataset. Keep this result for the report.\nSELECT\n  COUNT(*) AS total_events,\n  COUNT(DISTINCT user_pseudo_id) AS unique_players,\n  COUNT(DISTINCT event_date) AS days,\n  MIN(event_date) AS first_date,\n  MAX(event_date) AS last_date,\n  COUNTIF(user_pseudo_id IS NULL OR TRIM(user_pseudo_id) = '') AS missing_player_ids,\n  COUNTIF(event_timestamp IS NULL) AS missing_timestamps,\n  COUNTIF(event_name IS NULL OR TRIM(event_name) = '') AS missing_event_names\nFROM `firebase-public-project.analytics_153293282.events_2*`;\n", 'event_inventory': "-- First see what events we have and how many players we have for each event.\n-- Check the estimated amount of data BigQuery will read before running this.\nSELECT\n  event_name,\n  COUNT(*) AS event_count,\n  COUNT(DISTINCT user_pseudo_id) AS unique_players,\n  COUNTIF(user_pseudo_id IS NULL OR TRIM(user_pseudo_id) = '') AS missing_player_ids,\n  MIN(event_date) AS first_event_date,\n  MAX(event_date) AS last_event_date\nFROM `firebase-public-project.analytics_153293282.events_2*`\nGROUP BY event_name\nORDER BY event_count DESC;\n", 'event_parameter_inventory': '-- Find out what extra information is recorded with each event.\nSELECT\n  event_name,\n  param.key AS parameter_name,\n  COUNT(*) AS parameter_count,\n  COUNTIF(param.value.string_value IS NOT NULL) AS string_values,\n  COUNTIF(param.value.int_value IS NOT NULL) AS integer_values,\n  COUNTIF(param.value.float_value IS NOT NULL) AS float_values,\n  COUNTIF(param.value.double_value IS NOT NULL) AS double_values\nFROM `firebase-public-project.analytics_153293282.events_2*`,\nUNNEST(event_params) AS param\nGROUP BY event_name, parameter_name\nORDER BY event_name, parameter_count DESC;\n', 'retention_candidates': "-- A possible first training table. Run the inventory before using this.\n-- One row per player with an observed first_open and eight days of follow-up.\n-- Features use the first 24 hours. The target uses the next seven days.\nWITH events AS (\n  SELECT user_pseudo_id, event_name, event_timestamp\n  FROM `firebase-public-project.analytics_153293282.events_2*`\n  WHERE user_pseudo_id IS NOT NULL\n    AND TRIM(user_pseudo_id) != ''\n    AND event_timestamp IS NOT NULL\n),\nfirst_open AS (\n  SELECT user_pseudo_id, MIN(event_timestamp) AS first_open_us\n  FROM events\n  WHERE event_name = 'first_open'\n  GROUP BY user_pseudo_id\n),\ndata_end AS (\n  SELECT MAX(event_timestamp) AS last_event_us FROM events\n),\neligible_players AS (\n  SELECT\n    f.user_pseudo_id,\n    f.first_open_us,\n    f.first_open_us + 86400000000 AS cutoff_us,\n    f.first_open_us + 691200000000 AS label_end_us\n  FROM first_open AS f\n  CROSS JOIN data_end AS d\n  WHERE f.first_open_us + 691200000000 <= d.last_event_us\n)\nSELECT\n  p.user_pseudo_id,\n  p.first_open_us,\n  p.cutoff_us,\n  COUNTIF(e.event_timestamp < p.cutoff_us) AS early_events,\n  COUNTIF(e.event_name = 'session_start' AND e.event_timestamp < p.cutoff_us) AS early_sessions,\n  COUNTIF(e.event_name = 'level_start' AND e.event_timestamp < p.cutoff_us) AS early_level_starts,\n  COUNTIF(e.event_name = 'level_end' AND e.event_timestamp < p.cutoff_us) AS early_level_ends,\n  COUNTIF(e.event_name = 'post_score' AND e.event_timestamp < p.cutoff_us) AS early_scores_posted,\n  COUNTIF(e.event_name = 'session_start' AND e.event_timestamp >= p.cutoff_us) > 0 AS returned_next_7_days\nFROM eligible_players AS p\nJOIN events AS e\n  ON p.user_pseudo_id = e.user_pseudo_id\n  AND e.event_timestamp >= p.first_open_us\n  AND e.event_timestamp < p.label_end_us\nGROUP BY p.user_pseudo_id, p.first_open_us, p.cutoff_us\nORDER BY p.first_open_us, p.user_pseudo_id;\n"}

# Check estimated query sizes before running anything.
GIB = 1024 ** 3
TOTAL_LIMIT = 25 * GIB
estimates = {}

for name, sql in queries.items():
    config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=config)
    estimates[name] = job.total_bytes_processed
    print(f'{name}: {estimates[name] / GIB:.3f} GiB')

print(f'Total estimated scan: {sum(estimates.values()) / GIB:.3f} GiB')
if sum(estimates.values()) > TOTAL_LIMIT:
    raise RuntimeError('The estimated total is over 25 GiB. Stop and review the queries.')


In [ ]:
# Run the queries and save the results. The last query is a candidate
# retention dataset, so we only run it if the needed event types exist.
import csv
import json
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

if input('Using free or approved quota? Type RUN to export: ').strip() != 'RUN':
    raise RuntimeError('Export cancelled.')

# Use a new folder so results from an earlier run cannot get mixed in.
from tempfile import mkdtemp
folder = Path(mkdtemp(prefix='flood_it_audit_'))
remaining = TOTAL_LIMIT
manifest = {'exported_utc': datetime.now(timezone.utc).isoformat(), 'queries': []}
event_counts = {}

for name, sql in queries.items():
    if name == 'retention_candidates' and not all(event_counts.get(event, 0) > 0 for event in ('first_open', 'session_start')):
        print('Skipping retention: first_open or session_start is missing.')
        manifest['retention_skipped'] = True
        continue

    if estimates[name] > remaining or remaining <= 0:
        raise RuntimeError('Remaining scan limit is too small. Earlier CSVs remain in ' + str(folder))

    config = bigquery.QueryJobConfig(maximum_bytes_billed=remaining)
    job = client.query(sql, job_config=config)
    rows = job.result()
    columns = [field.name for field in rows.schema]
    count = 0

    with (folder / f'{name}.csv').open('w', newline='', encoding='utf-8') as output:
        writer = csv.writer(output)
        writer.writerow(columns)
        for row in rows:
            writer.writerow([row[column] for column in columns])
            count += 1
            if name == 'event_inventory':
                event_counts[row['event_name']] = row['event_count']

    # Keep track of billed bytes as well as the estimate to stay within the cap.
    billed = job.total_bytes_billed
    manifest['queries'].append({'name': name, 'job_id': job.job_id, 'rows': count, 'estimated_bytes': estimates[name], 'billed_bytes': billed, 'sql': sql})
    print(f'{name}: {count:,} rows saved')
    if billed is None:
        print('BigQuery did not report billed bytes. Stopping here so we can review the job.')
        break
    remaining -= max(billed, estimates[name])

(folder / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = folder / 'flood_it_audit.zip'
with ZipFile(archive, 'w') as bundle:
    for path in sorted(folder.iterdir()):
        if path.suffix in ('.csv', '.json'):
            bundle.write(path, arcname=path.name)

files.download(str(archive))


Upload `flood_it_audit.zip` to our project chat. We can review the real event counts, check the retention table and decide what model to train first. Keep the player-level CSV local and out of Git.